# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd
import numpy as np
from collections import Counter

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Pathing
from pathlib import Path

# Arithmetic
from math import sqrt

# Statistics
from scipy.stats import skew

# Make plots look nicer
sns.set_theme(style="whitegrid")
%matplotlib inline

## Load Flagged Flow Dataset for Analysis

In [ ]:
flagged_flows_path = Path("../data/processed/stage1_flagged_flows.csv")
flagged_flows = pd.read_csv(flagged_flows_path, low_memory=False)

# Sanity check everything flagged
print(flagged_flows["pred_anomaly"].value_counts())
print(flagged_flows["outcome"].value_counts())

# Show first few rows
display(flagged_flows.head())

# Check shape
print("Dataset shape:", flagged_flows.shape)

## Visualise Traffic Class Balance After Filtering

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x="Label", data=flagged_flows)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Count")
plt.title("Normal vs Attack Traffic")
plt.show()

# Numeric Features

## Candidate Features

In [ ]:
# Temporal decomposition
dt = pd.to_datetime(flagged_flows["Stime"], unit="s")
flagged_flows["hourofday"] = dt.dt.hour
flagged_flows["dayofweek"] = dt.dt.dayofweek

numeric_features = flagged_flows.select_dtypes(include=np.number).columns.tolist()
numeric_features = [feat for feat in numeric_features if feat not in ["Label", "pred_anomaly"]] # Exclude targets
print("Number of numeric features:", len(numeric_features))

## Compute Comprehensive Feature Summaries

In [ ]:
summary = []
for feat in numeric_features:
    tp_vals = flagged_flows.loc[flagged_flows["outcome"] == "TP", feat]
    fp_vals = flagged_flows.loc[flagged_flows["outcome"] == "FP", feat]

    # Compute Cohen's d
    mean_diff = tp_vals.mean() - fp_vals.mean()
    pooled_std = sqrt((tp_vals.std()**2 + fp_vals.std()**2)/2)
    cohen_d = mean_diff/pooled_std if pooled_std != 0 else 0

    # Summary stats
    tp_iqr = tp_vals.quantile(0.75) - tp_vals.quantile(0.25)
    fp_iqr = fp_vals.quantile(0.75) - fp_vals.quantile(0.25)
    tp_95, fp_95 = tp_vals.quantile(0.95), fp_vals.quantile(0.95)
    tp_5, fp_5 = tp_vals.quantile(0.05), fp_vals.quantile(0.05)
    tp_skew, fp_skew = skew(tp_vals, bias=False), skew(fp_vals, bias=False)

    # Tail differences
    tail_diff_95 = tp_95 - fp_95
    tail_diff_5 = tp_5 - fp_5

    # Skew difference
    skew_diff = tp_skew - fp_skew

    summary.append({
        "Feature": feat,
        "Cohen_d": cohen_d,
        "TP_median": tp_vals.median(),
        "FP_median": fp_vals.median(),
        "TP_IQR": tp_iqr,
        "FP_IQR": fp_iqr,
        "TP_95th": tp_95,
        "FP_95th": fp_95,
        "TP_5th": tp_5,
        "FP_5th": fp_5,
        "TP_skew": tp_skew,
        "FP_skew": fp_skew,
        "Tail_diff_95": tail_diff_95,
        "Tail_diff_5": tail_diff_5,
        "Skew_diff": skew_diff
    })
feature_summary = pd.DataFrame(summary)

# Screening lens
sorted_feature_summary = feature_summary.sort_values("Cohen_d", key=abs, ascending=False)
display(sorted_feature_summary)

## Visualise Top Feature Distributions

### Cohen's D (Mean Shift)

In [ ]:
sorted_by_cohen = feature_summary.sort_values("Cohen_d", key=abs, ascending=False)
top_cohen_features = sorted_by_cohen.head(7)["Feature"].tolist()
for feat in top_cohen_features:
    plt.figure(figsize=(6, 4))

    # Violin layer
    sns.violinplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        inner=None,
        alpha=0.5,
        dodge=False
    )

    # Box layer
    sns.boxplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        width=0.2,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        legend=False,
        dodge=False
    )

    plt.xlabel("Outcome")
    plt.ylabel(feat)
    plt.title(f"Distribution of {feat} by Outcome")
    plt.show()

### Combine Candidates

In [ ]:
# Top features by mean shift
top_cohen = feature_summary.sort_values("Cohen_d", key=abs, ascending=False).head(5)["Feature"].tolist()

# Top features by tail differences
feature_summary["Tail_diff_95_scaled"] = feature_summary["Tail_diff_95"] / (feature_summary["TP_IQR"] + feature_summary["FP_IQR"])
feature_summary["Tail_diff_95_scaled"] = feature_summary["Tail_diff_95_scaled"].replace([np.inf, -np.inf], np.nan)
top_tail_diff_95_scaled = feature_summary.sort_values("Tail_diff_95_scaled", key=abs, ascending=False).head(5)["Feature"].tolist()
feature_summary["Tail_diff_5_scaled"]  = feature_summary["Tail_diff_5"]  / (feature_summary["TP_IQR"] + feature_summary["FP_IQR"])
feature_summary["Tail_diff_5_scaled"]  = feature_summary["Tail_diff_5_scaled"].replace([np.inf, -np.inf], np.nan)
top_tail_diff_5_scaled = feature_summary.sort_values("Tail_diff_5_scaled", key=abs, ascending=False).head(5)["Feature"].tolist()

# Top features by skew difference
top_skew_diff = feature_summary.sort_values("Skew_diff", key=abs, ascending=False).head(5)["Feature"].tolist()

# Union of all candidates
all_top_features = top_cohen + top_tail_diff_95_scaled + top_tail_diff_5_scaled + top_skew_diff

# Sort features by frequency
feature_counts = Counter(all_top_features)
ranked_features = pd.DataFrame.from_dict(feature_counts, orient="index", columns=["Frequency"])
ranked_features = ranked_features.sort_values("Frequency", ascending=False)
ranked_features = ranked_features.merge(
    feature_summary[["Feature", "Cohen_d", "Tail_diff_95_scaled", "Tail_diff_5_scaled", "Skew_diff"]],
    left_index=True, right_on="Feature", how="left"
)
display(ranked_features)

# Select only the top 7 features
top_features = ranked_features.head(7)["Feature"].tolist()

### Layered Plots for Combined Candidate Set

In [ ]:
for feat in top_features:
    plt.figure(figsize=(6, 4))

    # Violin layer
    sns.violinplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        inner=None,
        alpha=0.5,
        dodge=False
    )

    # Box layer
    sns.boxplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        width=0.2,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        legend=False,
        dodge=False
    )

    plt.xlabel("Outcome")
    plt.ylabel(feat)
    plt.title(f"Distribution of {feat} by Outcome")
    plt.show()

## Plot Paired Feature Interactions

### Cohen's D (Mean Shift)

In [ ]:
sns.pairplot(
    flagged_flows,
    vars=top_cohen_features,
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    diag_kind="kde",
    corner=True
)
plt.suptitle("Top Feature Interactions by Outcome", y=1.02)
plt.show()

### Combined Candidate Feature Set

In [ ]:
sns.pairplot(
    flagged_flows,
    vars=top_features,
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    diag_kind="kde",
    corner=True,
    markers=["o", "s"],
    height=2.5
)
plt.suptitle("Top Feature Interactions by Outcome", y=1.02)
plt.show()

## Test Interaction Feature Effects

### Ratio Test

Tested whether collapsing `sttl` and `ttl_diff` into a ratio produced univariate separation between TP and FP flows. 

Result: Low Cohen's d and identical medians, indicating no meaningful marginal separation.

Conclusion: Separation appears to be interaction-based rather than reducible to a single scalar feature.

In [ ]:
sttl_over_ttl_diff = flagged_flows["sttl"]/(flagged_flows["ttl_diff"] + 1e-6)
tp_vals = sttl_over_ttl_diff[flagged_flows["outcome"] == "TP"]
fp_vals = sttl_over_ttl_diff[flagged_flows["outcome"] == "FP"]
mean_diff = tp_vals.mean() - fp_vals.mean()
pooled_std = np.sqrt((tp_vals.std()**2 + fp_vals.std()**2)/2)
cohen_d = mean_diff/pooled_std if pooled_std != 0 else 0
interaction_summary = pd.DataFrame({
    "Feature": ["sttl_over_ttl_diff"],
    "Cohen_d": [cohen_d],
    "TP_median": [tp_vals.median()],
    "FP_median": [fp_vals.median()],
    "TP_IQR": [tp_vals.quantile(0.75) - tp_vals.quantile(0.25)],
    "FP_IQR": [fp_vals.quantile(0.75) - fp_vals.quantile(0.25)]
})
display(interaction_summary)

### Product Test

Tested whether collapsing `sttl` and `ttl_diff` into a product produced univariate separation between TP and FP flows.

Result: Higher Cohen's d but still slightly weaker than `ttl_diff` alone.

Conclusion: The raw product doesn't capture any additional separation.

In [ ]:
sttl_x_ttl_diff = flagged_flows["sttl"]*flagged_flows["ttl_diff"]
tp_vals = sttl_x_ttl_diff[flagged_flows["outcome"] == "TP"]
fp_vals = sttl_x_ttl_diff[flagged_flows["outcome"] == "FP"]
mean_diff = tp_vals.mean() - fp_vals.mean()
pooled_std = np.sqrt((tp_vals.std()**2 + fp_vals.std()**2)/2)
cohen_d = mean_diff/pooled_std if pooled_std != 0 else 0
interaction_summary = pd.DataFrame({
    "Feature": ["sttl_x_ttl_diff"],
    "Cohen_d": [cohen_d],
    "TP_median": [tp_vals.median()],
    "FP_median": [fp_vals.median()],
    "TP_IQR": [tp_vals.quantile(0.75) - tp_vals.quantile(0.25)],
    "FP_IQR": [fp_vals.quantile(0.75) - fp_vals.quantile(0.25)]
})
display(interaction_summary)

Neither ratio nor product of `sttl` and `ttl_diff` produced stronger univariate separation than `ttl_diff` alone, suggesting that TP/FP separation is inherently 2-dimensional.

## Revisualise Using Other Plots

### Density Plots

#### STTL vs TTL Diff

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=flagged_flows,
    x="sttl",
    y="ttl_diff",
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    common_norm=False,
    alpha=0.8,
)
plt.xlabel("STTL")
plt.ylabel("TTL Diff")
plt.title("2D Density of STTL vs TTL Diff")
plt.show()

#### STTL vs Duration

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=flagged_flows,
    x="sttl",
    y="dur",
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    common_norm=False,
    alpha=0.8
)
plt.xlabel("STTL")
plt.ylabel("Duration")
plt.title("2D Density of STTL vs Duration")
plt.show()